### Experiment preparation

In [ ]:
import csv
import os

from QuantumLogistics import (
    DeltaCoarseningEngine,
    VRPExplorationsEncoder,
    VRPExplorationsSolver,
    Route,
    StandardRouteSolver,
)
from QuantumLogistics.LogisticsRoute.VrpRepGraph import vrpRepGraph
from QuantumLogistics.RouteSolver.Encoders.VRPExplorationsEncoder import VRPInstance

dataDir = os.path.join(os.getcwd(), "../../dataset")
resultsDir = os.path.join(os.getcwd(), "../../results")
csvOutputFilename = "knalecz_csvOutputFile.csv"

In [ ]:
# General CMT Details: {cmt numer : (number Trucks, optimal value)}
# http://vrp.atd-lab.inf.puc-rio.br/index.php/en/
CMTDetails = {
    "01": (5, 524.61),
    "02": (10, 835.26),
    "03": (8, 826.14),
    "04": (12, 1028.42),
    "05": (17, 1291.29),
    "11": (7, 1042.12),
    "12": (10, 819.56),
}

coarseningRate = 1.0
sampleSize = 1
plotOutputs = False
numberTrucks=2
cmtNum = "01"  # http://vrp.atd-lab.inf.puc-rio.br/index.php/en/plotted-instances?data=CMT1

cmtData = CMTDetails[cmtNum]
truckNumber = cmtData[0]
cmtFile = "CMT" + cmtNum + ".xml"

inputVect = {
    "cmtFile": cmtFile,
    "numberTrucks": truckNumber,
    "CMTBKS": cmtData[1],
    "coarseningRate": coarseningRate,
    "sampleSize": sampleSize,
    "verbose": plotOutputs,
}

# Resetting csv File
with open(os.path.join(resultsDir, csvOutputFilename), "w", newline="") as csv_file:
    writer = csv.writer(csv_file, delimiter=",")
    writer.writerow(
        [
            "cmtFileName",
            "Number of Trucks",
            "Coarsening Rate",
            "Run Number",
            "Solution Cost",
            "Relative Cost",
            "Solve Time",
        ]
    )

### Graph definition

In [ ]:
root = dataDir
file = os.path.join(root, cmtFile)
print(f"[i] Loading graph file {file}...")
LogisticsNetwork = vrpRepGraph(file)
LogisticsNetwork.generate_graph()

print(f"[i] Setting graph configurations for {file}...")
numberOfNodes = LogisticsNetwork.n
print(f"[i] Number of nodes {numberOfNodes} ...")

truckCapacity = max(LogisticsNetwork.nodeCapacities)
print(f"[i] Truck Capacity {truckCapacity} ...")

depot = LogisticsNetwork.nodelist.vehicle["arrival_node"]
print(f"[i] Depot Node {depot} ...")

### Route definiton

In [ ]:
# This defines the operating details for the route (i.e number of trucks, truck capacity (how many nodes can a truck go to))
routeConfig = {
    "vehicles": numberTrucks,
    "depot": depot,
    "truckCapacity": truckCapacity,
}  # Set to -1 for auto (will evenly distribute trucks)

# Define Coarsening Object/methods
# This is if coarsening is set to true (i.e the optimiser coarsens the graph before solving)
coarsenConfig = {"coarsenRate": coarseningRate, "radiusCoefficient": 0.2}
coarseningEngine = DeltaCoarseningEngine(**coarsenConfig)

### Solver definition

In [ ]:
encoder = VRPExplorationsEncoder()
solverAlg = VRPExplorationsSolver()
solver = StandardRouteSolver(encoder, solverAlg)
solverConfig = {"solver": "neal"}

### Solving

In [ ]:
runNumber = 0  # TODO docelowo dodać pętlę, jak w CMT_Experiment.py

LogisticsNetwork = vrpRepGraph(file)
LogisticsNetwork.generate_graph()
LogisticsNetwork.plotGraph()

route = Route(
    LogisticsNetwork, coarseningEngine=coarseningEngine, **routeConfig
)

In [ ]:
route.coarsen = (coarseningRate < 1)

solvedRoute, solveTime, cost = solver.solve(route, config=solverConfig)

In [ ]:
# Saving images
solutionImgFilename = f"{cmtFile}_{coarsenConfig['coarsenRate']}_{runNumber}_solution.png"
route.visualiseSolution(solvedRoute, saveImgFilepath=os.path.join(resultsDir, solutionImgFilename))

In [ ]:
# Output
solutionList = [
    cmtFile,
    numberTrucks,
    coarsenConfig["coarsenRate"],
    runNumber,
    cost,
    cost / CMTBKS,
    solveTime,
]

with open(os.path.join(resultsDir, csvOutputFilename), "a", newline="") as csv_file:
    writer = csv.writer(csv_file, delimiter=",")
    writer.writerow(solutionList)

### Comparison with "pure" VRP-experiments solvers results

In [ ]:
from VRP.quantum.BQM_based.full_qubo_solver import (
    FullQuboSolver as FQS,
    CapcFullQuboSolver as CFQS
)

vrp_instance = encoder.encode(route)
sps = FQS(
    vrp_instance.clients_num - 1,
    vrp_instance.vehicles_num,
    vrp_instance.cost_matrix
)
sps.solve(solver='neal')
sps.visualize(vrp_instance.xc, vrp_instance.yc)

In [ ]:
import numpy as np

routeSol = []
for i in range(sps.solution.shape[0]):
    var_list = np.transpose(sps.variables[i]).reshape(-1)
    sol_list = np.transpose(sps.solution[i]).reshape(-1)
    active_vars = [var_list[k] for k in range(len(var_list)) if sol_list[k] == 1]
    vehicle_route = [int(var.split('.')[2]) for var in active_vars]
    edgelist = [(0, vehicle_route[0])] + [(vehicle_route[j], vehicle_route[j + 1]) for j in range(len(vehicle_route) - 1)] + [(vehicle_route[-1], 0)]
    routeSol.append(edgelist)

print(routeSol)

In [ ]:
print(solvedRoute)